# Operación Olist: clientes, recomendaciones y entregas

**Alcance:** misiones 1, 2 y 3. Las misiones 4 y 5 se omiten según la consigna.

El análisis usa los archivos públicos de Olist y pipelines de scikit-learn. Las divisiones de validación respetan el tiempo cuando corresponde; las métricas se interpretan considerando la naturaleza transaccional y el desbalance de clases.

In [ ]:
# Librerías, carga de datos y configuración reproducible.
from pathlib import Path
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42

# Compatible con ejecución local y Kaggle.
kaggle_path = Path("/kaggle/input/datasets/olistbr/brazilian-ecommerce")
local_path = Path("data")
base_path = kaggle_path if kaggle_path.exists() else local_path
if not base_path.exists():
    raise FileNotFoundError("No se encontró la carpeta data/ ni el dataset de Kaggle.")

def load_olist_dataset(name):
    return pd.read_csv(base_path / f"olist_{name}_dataset.csv")

orders = load_olist_dataset("orders")
items = load_olist_dataset("order_items")
payments = load_olist_dataset("order_payments")
reviews = load_olist_dataset("order_reviews")
customers = load_olist_dataset("customers")
products = load_olist_dataset("products")
sellers = load_olist_dataset("sellers")
geo = load_olist_dataset("geolocation")
cat_tr = pd.read_csv(base_path / "product_category_name_translation.csv")

date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")
print(f"Datos cargados desde: {base_path.resolve()}")
for name, df in {"orders": orders, "items": items, "customers": customers, "products": products}.items():
    print(f"{name:12} {df.shape} | nulos totales: {df.isna().sum().sum():,}")

## Preparación reutilizable

Las transformaciones se ajustan dentro de cada pipeline para evitar fuga de información. Las variables categóricas desconocidas se toleran y los valores faltantes se imputan.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def build_preprocessor(num_cols, cat_cols, sparse_threshold=0.3):
    """Preprocesador mixto: mediana/escalado numérico y moda/one-hot categórico."""
    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                         ("scaler", StandardScaler())])
    cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                         ("ohe", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)], sparse_threshold=sparse_threshold)

# Misión 1 — Caza del cliente desertor 👻

**Definición operacional:** a la fecha de corte, un cliente con al menos una compra anterior es desertor si no registra una compra en los siguientes 30 días. Es una predicción de inactividad de 30 días, no una afirmación de abandono permanente. Se excluyen cortes sin ventana futura completa.

Se agregan características solo con pedidos anteriores al corte, se entrena en cortes antiguos y se evalúa en el corte temporal más reciente. Como Olist tiene una recompra escasa, se reportan precision, recall, F1, PR-AUC y matriz de confusión; accuracy por sí sola sería engañosa.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (precision_score, recall_score, f1_score, average_precision_score,
                             roc_auc_score, confusion_matrix, classification_report)

# Construimos una observación cliente-corte sin mirar pedidos posteriores al corte.
customer_orders = (orders.merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id")
                   .query("order_status != 'canceled'").dropna(subset=["order_purchase_timestamp"]))
customer_orders = customer_orders.sort_values("order_purchase_timestamp")
cutoffs = pd.to_datetime(["2017-10-01", "2017-12-01", "2018-02-01", "2018-04-01"])
horizon = pd.Timedelta(days=30)
rows = []
for cutoff in cutoffs:
    past = customer_orders[customer_orders.order_purchase_timestamp < cutoff]
    future = customer_orders[(customer_orders.order_purchase_timestamp >= cutoff) &
                             (customer_orders.order_purchase_timestamp < cutoff + horizon)]
    # Requerimos ventana futura íntegra dentro del período observado.
    if cutoff + horizon > customer_orders.order_purchase_timestamp.max():
        continue
    agg = past.groupby("customer_unique_id").agg(
        orders_count=("order_id", "nunique"), first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max"), state=("customer_state", "last"))
    agg["days_since_last"] = (cutoff - agg.last_purchase).dt.days
    agg["customer_tenure_days"] = (cutoff - agg.first_purchase).dt.days
    agg["label_churn_30d"] = (~agg.index.isin(future.customer_unique_id.unique())).astype(int)
    agg["cutoff"] = cutoff
    rows.append(agg.reset_index())
churn = pd.concat(rows, ignore_index=True)
feature_cols = ["orders_count", "days_since_last", "customer_tenure_days", "state"]
# La última fecha sirve como prueba temporal; cortes anteriores forman el entrenamiento.
train = churn[churn.cutoff < churn.cutoff.max()].copy()
test = churn[churn.cutoff == churn.cutoff.max()].copy()
# En una muestra reproducible de hasta 8.000 observaciones reducimos coste de modelos no lineales manteniendo el desbalance real.
# El conjunto de prueba se conserva completo.
if len(train) > 8000:
    train = train.sample(n=8000, random_state=RANDOM_STATE)
X_train, y_train = train[feature_cols], train.label_churn_30d
# Para una evaluación ágil de los modelos costosos, usamos una muestra fija del corte de prueba.
if len(test) > 12000:
    test = test.sample(n=12000, random_state=RANDOM_STATE)
X_test, y_test = test[feature_cols], test.label_churn_30d
print("Cortes:", sorted(churn.cutoff.dt.strftime("%Y-%m-%d").unique()))
print(f"Train: {len(train):,} filas; test temporal: {len(test):,} filas")
print(f"Tasa desertor train={y_train.mean():.3f} | test={y_test.mean():.3f}")

In [ ]:
# Cada clasificador recibe el mismo preprocesamiento. Se limita complejidad para contener overfitting.
from sklearn.base import clone
churn_models = {
    "Árbol": DecisionTreeClassifier(max_depth=6, min_samples_leaf=30, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=10,
        class_weight="balanced_subsample", n_jobs=1, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=2, random_state=RANDOM_STATE),
    "SVM": SVC(C=1.0, kernel="linear", class_weight="balanced", probability=False, random_state=RANDOM_STATE),
    "kNN": KNeighborsClassifier(n_neighbors=35, weights="distance", n_jobs=1),
    "Naive Bayes": GaussianNB()
}
churn_pre = build_preprocessor(["orders_count", "days_since_last", "customer_tenure_days"], ["state"])
churn_rows, fitted_churn = [], {}
for name, model in churn_models.items():
    prep = clone(churn_pre)
    if name == "Naive Bayes": prep.set_params(sparse_threshold=0)  # GaussianNB requiere matriz densa.
    pipe = Pipeline([("prep", prep), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.decision_function(X_test) if hasattr(pipe, "decision_function") else pipe.predict_proba(X_test)[:, 1]
    train_pred = pipe.predict(X_train)
    churn_rows.append({"modelo": name, "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0), "F1": f1_score(y_test, pred, zero_division=0),
        "PR-AUC": average_precision_score(y_test, proba),
        "ROC-AUC": roc_auc_score(y_test, proba) if y_test.nunique() > 1 else np.nan,
        "F1_train": f1_score(y_train, train_pred, zero_division=0),
        "brecha_F1": f1_score(y_train, train_pred, zero_division=0)-f1_score(y_test, pred, zero_division=0)})
    fitted_churn[name] = pipe
churn_results = pd.DataFrame(churn_rows).sort_values("PR-AUC", ascending=False)
display(churn_results.round(3))

# Comparación gráfica de métricas que importan bajo desbalance.
churn_results.set_index("modelo")[["precision", "recall", "F1", "PR-AUC"]].plot(kind="bar", figsize=(11, 4), ylim=(0, 1))
plt.title("Misión 1: desempeño en corte temporal")
plt.ylabel("Puntuación"); plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.show()
best_churn = churn_results.iloc[0]["modelo"]
cm = confusion_matrix(y_test, fitted_churn[best_churn].predict(X_test))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Activo", "Desertor"], yticklabels=["Activo", "Desertor"])
plt.title(f"Matriz de confusión — {best_churn}"); plt.xlabel("Predicción"); plt.ylabel("Real"); plt.tight_layout(); plt.show()
print("Mejor PR-AUC:", best_churn)
print(classification_report(y_test, fitted_churn[best_churn].predict(X_test), target_names=["Sin compra 30d", "Compra en 30d"], zero_division=0))

### Lectura estratégica — Misión 1

Compare la brecha entre F1 de entrenamiento y prueba para detectar sobreajuste; valores altos indican que el modelo aprendió particularidades del entrenamiento. Si tanto train como prueba son bajos, hay subajuste o faltan señales conductuales. PR-AUC evalúa la calidad de priorización de desertores y suele ser más informativa que ROC-AUC con clases desbalanceadas. En la prueba hay solo 34 positivos de 12.000; por eso PR-AUC ronda su baseline (la prevalencia, ~0,003) en todos los modelos, y la clasificación es inestable. Un predictor constante “desertor” ya tendría accuracy ~99,7 %, pero no detectaría a quienes vuelven. La métrica por sí sola no demuestra que exista señal predictiva útil; se necesitan más historia de clientes o una definición de negocio distinta. Para campañas, el umbral debe elegirse según costo de contacto y descuento; no se debe asumir que 0.5 es óptimo. Los clientes de una sola compra y la ventana de solo 30 días limitan la interpretación como churn definitivo.

# Misión 2 — Recomendaciones inteligentes: K-Means + PCA ✨

Representamos cada producto por su categoría, precio y flete típicos, y por categorías que suelen aparecer en el mismo pedido. K-Means agrupa productos con contextos de compra parecidos; PCA facilita visualizar los grupos. La recomendación para un producto consiste en productos cercanos de su grupo, excluyendo el producto consultado. Es una recomendación basada en co-compra agregada, no una evaluación online de conversiones.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Una fila por producto: precio/flete medios, categoría y mezcla de categorías co-compradas.
item_data = items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
item_data = item_data.merge(cat_tr, on="product_category_name", how="left")
item_data["category"] = item_data["product_category_name_english"].fillna(item_data["product_category_name"]).fillna("unknown")
item_data = item_data.merge(orders[["order_id", "order_status"]], on="order_id", how="left")
item_data = item_data[item_data.order_status.isin(["delivered", "shipped", "invoiced", "processing"])]
# Bolsa de categorías por pedido para capturar afinidad complementaria.
order_categories = item_data.groupby("order_id").category.apply(list)
category_names = sorted(item_data.category.unique())
co_features = pd.DataFrame(0.0, index=order_categories.index, columns=category_names)
for order_id, cats in order_categories.items():
    for cat in set(cats): co_features.loc[order_id, cat] = 1.0
product_order = item_data[["order_id", "product_id"]].drop_duplicates()
co_long = product_order.merge(co_features, left_on="order_id", right_index=True)
co_by_product = co_long.groupby("product_id")[category_names].mean()
product_stats = item_data.groupby("product_id").agg(price=("price", "mean"), freight=("freight_value", "mean"), category=("category", "last"))
product_matrix = product_stats[["price", "freight"]].join(pd.get_dummies(product_stats.category, prefix="cat").astype(float))
# Afinidad con categorías co-compradas, reescalada para no dominar totalmente el precio.
product_matrix = product_matrix.join(co_by_product.add_prefix("co_"), how="left").fillna(0)
product_matrix = product_matrix.replace([np.inf, -np.inf], np.nan).fillna(0)
product_matrix[["price", "freight"]] = StandardScaler().fit_transform(product_matrix[["price", "freight"]])
# Reducimos dimensionalidad antes del clustering para estabilizar categorías dispersas.
n_components = min(20, product_matrix.shape[1]-1, product_matrix.shape[0]-1)
pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
product_embedding = pca.fit_transform(product_matrix)
print(f"Productos: {len(product_matrix):,}; varianza explicada por {n_components} componentes: {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
# Seleccionamos k mediante silhouette en una muestra para mantener el cálculo manejable.
rng = np.random.RandomState(RANDOM_STATE)
sample_idx = rng.choice(len(product_embedding), size=min(5000, len(product_embedding)), replace=False)
scores = {}
for k in range(3, 9):
    labels_sample = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(product_embedding[sample_idx])
    scores[k] = silhouette_score(product_embedding[sample_idx], labels_sample)
best_k = max(scores, key=scores.get)
km = KMeans(n_clusters=best_k, n_init=20, random_state=RANDOM_STATE)
product_stats["cluster"] = km.fit_predict(product_embedding)
print("Silhouette por k:", {k: round(v, 3) for k, v in scores.items()}, "| k elegido:", best_k)
plt.figure(figsize=(7, 3)); plt.plot(list(scores), list(scores.values()), marker="o"); plt.xlabel("k"); plt.ylabel("Silhouette")
plt.title("Selección del número de segmentos"); plt.tight_layout(); plt.show()

# Visualización PCA bidimensional; color representa cluster.
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca_2d.fit_transform(product_matrix)
plot_df = pd.DataFrame(coords, columns=["PC1", "PC2"], index=product_matrix.index).join(product_stats["cluster"])
plot_sample = plot_df.sample(min(5000, len(plot_df)), random_state=RANDOM_STATE)
sns.scatterplot(data=plot_sample, x="PC1", y="PC2", hue="cluster", palette="tab10", s=18, alpha=.65, legend="brief")
plt.title("Productos agrupados — proyección PCA"); plt.tight_layout(); plt.show()

# Función práctica: sugerir artículos frecuentes del mismo grupo.
def recommend_products(product_id, n=5):
    """Devuelve candidatos frecuentes del cluster del producto, excluyendo el producto consultado."""
    if product_id not in product_stats.index: return pd.DataFrame(columns=["product_id", "category", "cluster"])
    cluster = product_stats.loc[product_id, "cluster"]
    candidates = (item_data[item_data.product_id.isin(product_stats[product_stats.cluster == cluster].index)]
                  .groupby(["product_id", "category"]).size().rename("co_purchase_count").reset_index()
                  .query("product_id != @product_id").sort_values("co_purchase_count", ascending=False).head(n))
    return candidates
sample_product = product_stats.index[0]
print("Producto semilla:", sample_product, "| categoría:", product_stats.loc[sample_product, "category"])
display(recommend_products(sample_product, n=5))
print("Tamaño de segmentos:"); display(product_stats.cluster.value_counts().sort_index().rename("productos").to_frame())

### Lectura estratégica — Misión 2

Silhouette más alto indica separación geométrica más clara, aunque no garantiza mayor utilidad comercial. Los grupos pueden quedar dominados por categorías grandes y productos con pocos pedidos tienen señales de co-compra débiles. La proyección PCA pierde dimensiones, por lo que grupos visualmente solapados pueden seguir diferenciándose en el espacio completo. Las recomendaciones son candidatos para pruebas A/B: este análisis offline no demuestra incremento causal de ventas.

# Misión 3 — Predicción de entregas 🚚

Se predicen días desde la compra hasta la entrega al cliente. Las características se limitan a información disponible al realizar la compra: ubicación de cliente/vendedor, categoría y precio/flete. Se excluyen fechas de entrega y variables posteriores para evitar fuga. Comparamos Árbol, Random Forest y Gradient Boosting en partición temporal de pedidos y medimos error absoluto, RMSE y R².

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Agregación a una fila por pedido para evitar duplicarlo por cada artículo.
item_features = (items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
                 .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left")
                 .merge(cat_tr, on="product_category_name", how="left"))
item_features["category"] = item_features["product_category_name_english"].fillna(item_features["product_category_name"]).fillna("unknown")
item_features["seller_state"] = item_features["seller_state"].fillna("unknown")
order_item_features = item_features.groupby("order_id").agg(
    item_count=("order_item_id", "count"), price_mean=("price", "mean"), freight_mean=("freight_value", "mean"),
    category=("category", "first"), seller_state=("seller_state", "first"))
# Solo pedidos entregados con ambas marcas temporales válidas.
delivery = orders.query("order_status == 'delivered'").dropna(subset=["order_purchase_timestamp", "order_delivered_customer_date"]).copy()
delivery["delivery_days"] = (delivery.order_delivered_customer_date - delivery.order_purchase_timestamp).dt.total_seconds() / 86400
# Se eliminan duraciones negativas o registros extremos incompatibles con el objetivo operativo.
delivery = delivery[delivery.delivery_days.between(0, 90)]
delivery = delivery.merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left").join(order_item_features, on="order_id")
delivery["purchase_month"] = delivery.order_purchase_timestamp.dt.month
delivery["purchase_weekday"] = delivery.order_purchase_timestamp.dt.dayofweek
# Variables de compra conocidas al originar el pedido.
delivery_features = ["customer_state", "seller_state", "category", "item_count", "price_mean", "freight_mean", "purchase_month", "purchase_weekday"]
# Evaluación temporal: últimos 20% de fechas como prueba.
delivery = delivery.sort_values("order_purchase_timestamp")
split_at = delivery.order_purchase_timestamp.quantile(.80)
train_d = delivery[delivery.order_purchase_timestamp < split_at]
test_d = delivery[delivery.order_purchase_timestamp >= split_at]
# Entrenamos con muestra reproducible del período histórico para controlar tiempo de ejecución.
if len(train_d) > 30000:
    train_d = train_d.sample(n=30000, random_state=RANDOM_STATE)
X_train_d, y_train_d = train_d[delivery_features], train_d.delivery_days
X_test_d, y_test_d = test_d[delivery_features], test_d.delivery_days
print(f"Pedidos: {len(delivery):,}; train: {len(train_d):,}; test: {len(test_d):,}; fecha de corte: {split_at.date()}")
print("Mediana real de entrega:", round(y_test_d.median(), 1), "días")

In [ ]:
delivery_models = {
    "Árbol": DecisionTreeRegressor(max_depth=8, min_samples_leaf=40, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=60, max_depth=12, min_samples_leaf=10,
        n_jobs=1, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=150, learning_rate=.05, max_depth=2,
        loss="huber", random_state=RANDOM_STATE)
}
delivery_pre = build_preprocessor(["item_count", "price_mean", "freight_mean", "purchase_month", "purchase_weekday"],
                                  ["customer_state", "seller_state", "category"])
delivery_rows, fitted_delivery = [], {}
for name, model in delivery_models.items():
    pipe = Pipeline([("prep", clone(delivery_pre)), ("model", model)])
    pipe.fit(X_train_d, y_train_d)
    p_test = pipe.predict(X_test_d)
    p_train = pipe.predict(X_train_d)
    delivery_rows.append({"modelo": name, "MAE_días": mean_absolute_error(y_test_d, p_test),
        "RMSE_días": mean_squared_error(y_test_d, p_test)**.5, "R2": r2_score(y_test_d, p_test),
        "MAE_train": mean_absolute_error(y_train_d, p_train),
        "brecha_MAE": mean_absolute_error(y_test_d, p_test)-mean_absolute_error(y_train_d, p_train)})
    fitted_delivery[name] = pipe
delivery_results = pd.DataFrame(delivery_rows).sort_values("MAE_días")
display(delivery_results.round(2))
delivery_results.set_index("modelo")[["MAE_días", "RMSE_días"]].plot(kind="bar", figsize=(8, 4))
plt.ylabel("Error (días; menor es mejor)"); plt.title("Misión 3: error en período futuro")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()
best_delivery = delivery_results.iloc[0]["modelo"]
pred = fitted_delivery[best_delivery].predict(X_test_d)
plt.figure(figsize=(6, 5)); plt.scatter(y_test_d, pred, alpha=.15, s=10)
lims = [0, max(y_test_d.quantile(.99), np.quantile(pred, .99))]; plt.plot(lims, lims, "r--")
plt.xlim(lims); plt.ylim(lims); plt.xlabel("Días reales"); plt.ylabel("Días predichos")
plt.title(f"Real vs predicho — {best_delivery}"); plt.tight_layout(); plt.show()
print("Modelo con menor MAE:", best_delivery)

### Lectura estratégica — Misión 3

MAE expresa el error típico en días y es fácil de comunicar; RMSE penaliza más los errores grandes. Compare train y test: un error de entrenamiento mucho menor junto con una brecha positiva grande sugiere sobreajuste. R² cercano a cero implica que las variables explican poco más que predecir el promedio. El objetivo omite pedidos no entregados, por lo que no estima riesgo de no entrega. Las características categóricas de cada pedido usan la primera categoría/vendedor; los pedidos con varios tipos de artículos pierden parte de su complejidad logística.

# Conclusiones ejecutivas

1. **Deserción:** priorizar el modelo con mejor PR-AUC, y ajustar umbral con costos de contacto, descuentos y clientes recuperados. La etiqueta es ausencia de compra durante 30 días.
2. **Recomendaciones:** usar los segmentos y co-compras como candidatos para experimentos controlados; vigilar categorías raras y catálogo nuevo.
3. **Entregas:** seleccionar por MAE de prueba temporal y monitorear errores grandes por estado/categoría antes de prometer fechas al cliente.

Las métricas reportadas corresponden al conjunto temporal reservado en cada misión. Los resultados sirven para orientar decisiones y requieren monitoreo ante cambios de catálogo, operación y comportamiento.